In [0]:
# ========================================
# Silver Layer: Regions with Data Quality
# ========================================

from pyspark.sql.functions import col, trim, upper, current_timestamp, when, length, monotonically_increasing_id

# Read from Bronze
df_bronze = spark.read.table("ecommerce.e_comm_bronze.tblregions")

print(f"Bronze record count: {df_bronze.count()}")

# ── Data Quality Step 1: Standardize and clean text ──
df_cleaned = df_bronze \
    .withColumn("region_id", trim(col("region_id"))) \
    .withColumn("region", trim(upper(col("region"))))

# ── Data Quality Step 2: Remove duplicates ──
df_deduped = df_cleaned.dropDuplicates(["region_id"])
df_deduped.createOrReplaceTempView("vw_df_deduped")
print(f"After deduplication: {df_deduped.count()}")

# ── Data Quality Step 3: Add surrogate_key and data quality flag ──
df_with_dq = spark.sql("""
        select 
        row_number() over (order by region_id) as region_key,
        region_id,
        region as region_name, 
        case when region is not null or trim(region)!=""
            then "is_valid"
            else null end as dq_note ,
        current_timestamp() as load_ts
        from vw_df_deduped 
                       """)


# # ── Data Quality Step 4: Add surrogate key ──
# df_silver_region = df_deduped \
#     .withColumn("region_key", monotonically_increasing_id() + 1) \
#     .withColumn("load_ts", current_timestamp()) \
#     .select("region_key", "region_id", "region", "is_valid", "is_null_or_empty", "dq_check_ts", "load_ts")

# Show data quality summary
print("\n=== Data Quality Summary ===")
print(f"Total records: {df_with_dq.count()}")
print(f"Valid records: {df_with_dq.filter(col('dq_note') == "is_valid").count()}")
print(f"Invalid records: {df_with_dq.filter(col('dq_note') != "is_valid").count()}")

# Show sample of invalid records if any exist
invalid_records = df_with_dq.filter(col('dq_note') != "is_valid")
if invalid_records.count() > 0:
    print("\nSample invalid records:")
    invalid_records.show(5, truncate=False)

# Create temp view for querying
df_with_dq.createOrReplaceTempView("vw_regions")

print("\n✅ Silver transformation complete with data quality checks")

In [0]:
#write data to a delta table
df_with_dq \
    .write \
        .format("delta") \
            .option("overwriteSchema", "true") \
                .mode("overwrite").saveAsTable("ecommerce.e_comm_silver.regions")